# Feed Forward Network with PyTorch

## AI Usage

I used AI to investigate and understand errors. I also used copilot autocomplete here and there for variables names and working with list comprehensions.

## PyTorch setup

In [260]:
import torch

In [261]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load data and create Datasets

In [262]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [263]:
transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [264]:
from torch.utils.data import Subset

Split the training data into training and validation sets

In [265]:
train_indices = list(range(50000))
val_indices = list(range(50000, len(train_data)))

val_data = Subset(train_data, val_indices)
train_data = Subset(train_data, train_indices)
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

Training set size: 50000
Validation set size: 10000


In [266]:
test_data = MNIST(root="data", train=False, download=True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the Neural Network

In [267]:
import torch.nn as nn

In [268]:
input_size = train_data[0][0].numel()

# I think to(device) is not needed here but I added it just in case
model1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model3 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

## Defining the optimizer

In [269]:
import torch.optim as optim

In [270]:
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)

## Defining loss function

In [271]:
criterion = nn.CrossEntropyLoss()

## Defining mini-batches creation function

In [272]:
import numpy as np

In [273]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train loop

In [274]:
from tqdm import tqdm

In [275]:
def train(model, train_data, optimizer, criterion, epochs, batch_size, patience=None, delta=0.005):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Accuracy
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)
        epoch_train_accuracy = train_correct / train_total

        # Eval
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # Accuracy
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)
        epoch_val_accuracy = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Train Acc: {epoch_train_accuracy:.4f}, Val Acc: {epoch_val_accuracy:.4f}")

        # Early stopping
        if patience is not None:
            if epoch == 0:
                best_val_loss = epoch_val_loss
                patience_counter = 0
            else:
                if epoch_val_loss < best_val_loss - delta:
                    best_val_loss = epoch_val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stop triggered")
                break

    return epoch_train_loss, epoch_val_loss, epoch_train_accuracy, epoch_val_accuracy

In [276]:
model1_trained = train(model1, train_data, optimizer1, criterion, epochs=30, batch_size=64, patience=5)

Epoch 1/30: 100%|██████████| 782/782 [00:03<00:00, 232.53it/s]


Epoch [1/30], Loss: 0.3931, Val Loss: 0.2011, Train Acc: 0.8958, Val Acc: 0.9456


Epoch 2/30: 100%|██████████| 782/782 [00:03<00:00, 243.79it/s]


Epoch [2/30], Loss: 0.1891, Val Loss: 0.1478, Train Acc: 0.9460, Val Acc: 0.9602


Epoch 3/30: 100%|██████████| 782/782 [00:03<00:00, 241.52it/s]


...

Epoch [10/30], Loss: 0.0285, Val Loss: 0.0858, Train Acc: 0.9932, Val Acc: 0.9768


Epoch 11/30: 100%|██████████| 782/782 [00:03<00:00, 240.59it/s]


Epoch [11/30], Loss: 0.0231, Val Loss: 0.0875, Train Acc: 0.9952, Val Acc: 0.9771


Epoch 12/30: 100%|██████████| 782/782 [00:03<00:00, 238.10it/s]


Epoch [12/30], Loss: 0.0184, Val Loss: 0.0900, Train Acc: 0.9967, Val Acc: 0.9778
Early stop triggered


In [277]:
model2_trained = train(model2, train_data, optimizer2, criterion, epochs=20, batch_size=86, patience=3)

Epoch 1/20: 100%|██████████| 582/582 [00:03<00:00, 170.22it/s]


Epoch [1/20], Loss: 0.2744, Val Loss: 0.1658, Train Acc: 0.9178, Val Acc: 0.9535


Epoch 2/20: 100%|██████████| 582/582 [00:04<00:00, 141.93it/s]


Epoch [2/20], Loss: 0.1505, Val Loss: 0.1418, Train Acc: 0.9563, Val Acc: 0.9618


Epoch 3/20: 100%|██████████| 582/582 [00:02<00:00, 239.77it/s]


...

Epoch [5/20], Loss: 0.1031, Val Loss: 0.1383, Train Acc: 0.9723, Val Acc: 0.9727


Epoch 6/20: 100%|██████████| 582/582 [00:02<00:00, 221.13it/s]


Epoch [6/20], Loss: 0.0966, Val Loss: 0.1326, Train Acc: 0.9733, Val Acc: 0.9715


Epoch 7/20: 100%|██████████| 582/582 [00:02<00:00, 227.41it/s]


Epoch [7/20], Loss: 0.0778, Val Loss: 0.1531, Train Acc: 0.9779, Val Acc: 0.9716
Early stop triggered


In [278]:
model3_trained = train(model3, train_data, optimizer3, criterion, epochs=35, batch_size=32, patience=7)

Epoch 1/35: 100%|██████████| 1563/1563 [00:05<00:00, 266.33it/s]


Epoch [1/35], Loss: 2.2738, Val Loss: 2.2357, Train Acc: 0.2261, Val Acc: 0.3682


Epoch 2/35: 100%|██████████| 1563/1563 [00:05<00:00, 266.15it/s]


Epoch [2/35], Loss: 2.1774, Val Loss: 2.0918, Train Acc: 0.4529, Val Acc: 0.5425


Epoch 3/35: 100%|██████████| 1563/1563 [00:04<00:00, 341.09it/s]


...

Epoch [33/35], Loss: 0.2955, Val Loss: 0.2738, Train Acc: 0.9151, Val Acc: 0.9230


Epoch 34/35: 100%|██████████| 1563/1563 [00:05<00:00, 294.31it/s]


Epoch [34/35], Loss: 0.2920, Val Loss: 0.2709, Train Acc: 0.9161, Val Acc: 0.9236


Epoch 35/35: 100%|██████████| 1563/1563 [00:05<00:00, 281.28it/s]


Epoch [35/35], Loss: 0.2887, Val Loss: 0.2680, Train Acc: 0.9169, Val Acc: 0.9244


In [279]:
for model, (train_loss, val_loss, train_accuracy, val_accuracy) in zip(
    ["Model 1", "Model 2", "Model 3"],
    [model1_trained, model2_trained, model3_trained]
):
    print(f"{model} - Final Training Loss: {train_loss:.4f}, Final Validation Loss: {val_loss:.4f}, Training Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}")

Model 1 - Final Training Loss: 0.0184, Final Validation Loss: 0.0900, Training Accuracy: 0.9967, Validation Accuracy: 0.9778
Model 2 - Final Training Loss: 0.0778, Final Validation Loss: 0.1531, Training Accuracy: 0.9779, Validation Accuracy: 0.9716
Model 3 - Final Training Loss: 0.2887, Final Validation Loss: 0.2680, Training Accuracy: 0.9169, Validation Accuracy: 0.9244


## Models evaluation

In [282]:
def evaluate_model(model, dataset, criterion, batch_size=256):
    x_data = [x for x, _ in dataset]
    y_data = [y for _, y in dataset]

    model.eval()
    total_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in create_minibatches(batch_size, x_data, y_data):
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return {
        "test_loss": total_loss / test_total,
        "test_accuracy": test_correct / test_total,
    }


models = {
    "Model 1": model1,
    "Model 2": model2,
    "Model 3": model3,
}

results = {}
for name, model in models.items():
    metrics = evaluate_model(model, test_data, criterion, batch_size=256)
    results[name] = metrics
    print(
        f"{name} = Test Loss: {metrics['test_loss']:.4f}, Test Accuracy: {metrics['test_accuracy']:.4f}"
    )

# Get the best model
best_name = max(results, key=lambda k: results[k]["test_accuracy"])
best_metrics = results[best_name]

print("\nBest model:", best_name)
print(
    f"{best_name} Test Loss: {best_metrics['test_loss']:.4f}, {best_name} Test Accuracy: {best_metrics['test_accuracy']:.4f}"
)

Model 1 = Test Loss: 0.0891, Test Accuracy: 0.9734
Model 2 = Test Loss: 0.1380, Test Accuracy: 0.9711
Model 3 = Test Loss: 0.2776, Test Accuracy: 0.9215

Best model: Model 1
Model 1 Test Loss: 0.0891, Model 1 Test Accuracy: 0.9734


## Reflection

This activity helped me to have a better understanding of the architecture and flow of a neural network. I learned how to use pythorch datasets and dataloaders and why we have to use dataloaders. I also gained better understanding of basic things like the calculaton of the loss and accuracy for each epoch. in conclusion I liked this activity because I learned how to do things withou libraries and I think I can improve my data processing or training function because I feel like my models were a little slow.